<a href="https://colab.research.google.com/github/aycaaozturk/AML-project/blob/main/AML_clinical_sample_SPLIT_IMPUTATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# AML CLINICAL SAMPLE DATA:
# PATIENT-ALIGNED TRAIN / VALIDATION / TEST SPLIT + IMPUTATION
#
# Tailored to:
# REDUCED_clinical_sample_missingness_checked.csv
#
# IMPORTANT:
# - This table contains multiple samples for some patients.
# - Therefore, sample rows must be split by PATIENT_ID, not independently.
# - Preferred approach: reuse the patient IDs from the already-created
#   clinical-patient train/validation/test files so both tables stay aligned.
# ============================================================

# -------------------------
# 0. Imports and settings
# -------------------------
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42

# -------------------------
# 1. Paths
# -------------------------
SAMPLE_INPUT_PATH = (
    '/content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Sample/'
    'REDUCED_clinical_sample_missingness_checked.csv'
)

# These are the patient-level split files created by the previous notebook.
PATIENT_SPLIT_DIR = Path(
    '/content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Patient/'
    'split_and_imputed'
)

OUTPUT_DIR = Path(
    '/content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Sample/'
    'sample_split_and_imputed'
)

# Preferred: use exactly the same patient membership as the clinical-patient
# data. Set to False only if those files are not available.

#Use exactly the same patient membership as the previously created clinical-patient splits.
ALIGN_WITH_PATIENT_SPLITS = True

# Variables with missingness above this threshold are dropped from the
# modeling feature set. They remain documented in metadata.
EXCESSIVE_MISSINGNESS_THRESHOLD = 0.80

# For high-missing categorical biomarkers, preserving an explicit
# "Missing" category can be more honest than replacing them with the mode.
CATEGORICAL_MISSING_LABEL = 'Missing'

#This is sometimes better than replacing it with the most frequent biological category.
#For example, replacing a missing genetic test with No could falsely imply
#that the alteration was tested and not detected.
#An explicit Missing category preserves the distinction.


In [ ]:
# -------------------------
# 2. Load sample-level data
# -------------------------
df = pd.read_csv(
    SAMPLE_INPUT_PATH,
    na_values=['', ' ', 'NA', 'N/A', 'NaN']
)

print('Original sample dataset shape:', df.shape)
print('Unique patients:', df['PATIENT_ID'].nunique())
print('Unique samples:', df['SAMPLE_ID'].nunique())

# Standardize missing-like values.
MISSING_LIKE_VALUES = [
    'Unknown',
    'Not done',
    'Unevaluable',
    '.'
]
df = df.replace(MISSING_LIKE_VALUES, np.nan)

# Remove leading/trailing spaces from text values.
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].apply(
        lambda value: value.strip() if isinstance(value, str) else value
    )

# Required identifiers.
ID_COLS = ['PATIENT_ID', 'SAMPLE_ID']
missing_id_cols = [col for col in ID_COLS if col not in df.columns]
if missing_id_cols:
    raise KeyError(f'Missing required identifier columns: {missing_id_cols}')

# Each sample ID should identify one row.
if df['SAMPLE_ID'].duplicated().any():
    duplicated = df.loc[
        df['SAMPLE_ID'].duplicated(keep=False), 'SAMPLE_ID'
    ].tolist()
    raise ValueError(
        'Duplicate SAMPLE_ID values found. Resolve before preprocessing: '
        f'{duplicated[:20]}'
    )



Original sample dataset shape: (1025, 31)
Unique patients: 899
Unique samples: 1025


In [ ]:
# -------------------------
# 3. Define feature roles
# -------------------------
# These columns are identifiers or study metadata, not biological predictors
# (inputs to predict stuff).
# They are retained in output files but excluded from feature imputation/modeling.

#A constant variable cannot help distinguish patients.

METADATA_COLS = [
    'ONCOTREE_CODE',
    'ANALYSIS_COHORT',
    'CANCER_TYPE',  #all leukamia, aml
    'CANCER_TYPE_DETAILED',
    'SOMATIC_STATUS'
]
METADATA_COLS = [col for col in METADATA_COLS if col in df.columns]

# Binary genomic indicators stored numerically as 0/1.
#binary genomic indicators, not continuous measurements.
BINARY_FEATURE_COLS = [
    'T_6_9',
    'T_8_21',
    'T_3_5_Q25_Q34',
    'T_6_11_Q27_Q23',
    'T_9_11_P22_Q23',
    'T_10_11_P11_2_Q23',
    'T_11_19_Q23_P13_1',
    'INV_16',
    'DEL_5Q',
    'DEL_7Q',
    'DEL_9Q',
    'MONOSOMY_5',
    'MONOSOMY_7',
    'TRISOMY_8',
    'TRISOMY_21',
    'MLL',
    'MINUS_Y',
    'MINUS_X',
    'FLT3_ITD_POSITIVE',
    'FLT3_PM',
    'NPM_MUTATION',
    'CEBPA_MUTATION',
    'WT1_MUTATION',
    'NUP98_NSD1'
]
BINARY_FEATURE_COLS = [col for col in BINARY_FEATURE_COLS if col in df.columns]

# Continuous feature. It is biologically conditional on FLT3-ITD positivity.
CONDITIONAL_CONTINUOUS_COL = 'FLT3_ITD_ALLELIC_RATIO'

# Categorical biomarker.
#If it is missing, the code later assigns the explicit label:
#Missing
#instead of pretending that the most common biological result was observed.

CATEGORICAL_FEATURE_COLS = [
    'CBFT2A3_GLIS'
]
CATEGORICAL_FEATURE_COLS = [
    col for col in CATEGORICAL_FEATURE_COLS if col in df.columns
]

# Validate that binary columns contain only 0, 1, and missing.
for col in BINARY_FEATURE_COLS:
    observed = set(df[col].dropna().unique().tolist())
    if not observed.issubset({0, 1, 0.0, 1.0}):
        raise ValueError(
            f'{col} contains unexpected values: {sorted(observed)}'
        )


In [ ]:
# -------------------------
# 4. Drop excessively sparse modeling features
# -------------------------
# This decision is based on the reduced dataset's data quality and is recorded.
feature_candidates = (
    BINARY_FEATURE_COLS
    + ([CONDITIONAL_CONTINUOUS_COL]
       if CONDITIONAL_CONTINUOUS_COL in df.columns else [])
    + CATEGORICAL_FEATURE_COLS
)

missing_fraction = df[feature_candidates].isna().mean()
EXCESSIVELY_MISSING_COLS = missing_fraction[
    missing_fraction > EXCESSIVE_MISSINGNESS_THRESHOLD
].index.tolist()

print('\nColumns excluded for excessive missingness:')
print(EXCESSIVELY_MISSING_COLS)

# In this dataset, NUP98_NSD1 is expected to be excluded by the 80% rule.
# FLT3_ITD_ALLELIC_RATIO is handled specially below, so do not drop it merely
# because it is structurally unavailable for FLT3-negative samples.
EXCESSIVELY_MISSING_COLS = [
    col for col in EXCESSIVELY_MISSING_COLS
    if col != CONDITIONAL_CONTINUOUS_COL
]

BINARY_FEATURE_COLS = [
    col for col in BINARY_FEATURE_COLS
    if col not in EXCESSIVELY_MISSING_COLS
]
CATEGORICAL_FEATURE_COLS = [
    col for col in CATEGORICAL_FEATURE_COLS
    if col not in EXCESSIVELY_MISSING_COLS
]



Columns excluded for excessive missingness:
['FLT3_ITD_ALLELIC_RATIO']


In [ ]:
# -------------------------
# 5. Create patient-aligned splits
# -------------------------
def read_patient_ids(path):
    patient_df = pd.read_csv(path, usecols=['PATIENT_ID'])
    return set(patient_df['PATIENT_ID'].astype(str))


if ALIGN_WITH_PATIENT_SPLITS:
    # Raw patient split files are preferred because they preserve the exact
    # patient membership independently of later imputation/modeling choices.
    patient_train_path = PATIENT_SPLIT_DIR / 'train_raw.csv'
    patient_val_path = PATIENT_SPLIT_DIR / 'validation_raw.csv'
    patient_test_path = PATIENT_SPLIT_DIR / 'test_raw.csv'

    required_paths = [
        patient_train_path,
        patient_val_path,
        patient_test_path
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Patient split files were not found. Either correct '
            'PATIENT_SPLIT_DIR or set ALIGN_WITH_PATIENT_SPLITS=False. '
            f'Missing: {missing_paths}'
        )

    train_patient_ids = read_patient_ids(patient_train_path)
    val_patient_ids = read_patient_ids(patient_val_path)
    test_patient_ids = read_patient_ids(patient_test_path)

    # Ensure patient sets do not overlap.
    assert train_patient_ids.isdisjoint(val_patient_ids)
    assert train_patient_ids.isdisjoint(test_patient_ids)
    assert val_patient_ids.isdisjoint(test_patient_ids)

    train_df = df[df['PATIENT_ID'].isin(train_patient_ids)].copy()
    val_df = df[df['PATIENT_ID'].isin(val_patient_ids)].copy()
    test_df = df[df['PATIENT_ID'].isin(test_patient_ids)].copy()

    sample_patients = set(df['PATIENT_ID'])
    assigned_patients = (
        train_patient_ids | val_patient_ids | test_patient_ids
    )
    unassigned_sample_patients = sorted(sample_patients - assigned_patients)

    if unassigned_sample_patients:
        print(
            '\nWarning: sample rows from patients absent from the patient-level '
            'survival splits were not assigned:'
        )
        print(unassigned_sample_patients[:20])

else:
    # Fallback: group-based split. No patient's samples can cross splits.
    splitter_1 = GroupShuffleSplit(
        n_splits=1,
        test_size=0.30,
        random_state=RANDOM_STATE
    )
    train_idx, temp_idx = next(
        splitter_1.split(df, groups=df['PATIENT_ID'])
    )

    train_df = df.iloc[train_idx].copy()
    temp_df = df.iloc[temp_idx].copy()

    splitter_2 = GroupShuffleSplit(
        n_splits=1,
        test_size=0.50,
        random_state=RANDOM_STATE
    )
    val_idx, test_idx = next(
        splitter_2.split(temp_df, groups=temp_df['PATIENT_ID'])
    )

    val_df = temp_df.iloc[val_idx].copy()
    test_df = temp_df.iloc[test_idx].copy()
    unassigned_sample_patients = []


# Verify that no patient appears in more than one split.
train_patients = set(train_df['PATIENT_ID'])
val_patients = set(val_df['PATIENT_ID'])
test_patients = set(test_df['PATIENT_ID'])

assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)

print('\nSplit summary:')
for name, split_df in [
    ('Training', train_df),
    ('Validation', val_df),
    ('Test', test_df)
]:
    print(
        f'{name}: {len(split_df)} samples, '
        f'{split_df["PATIENT_ID"].nunique()} patients'
    )


['TARGET-20-PAPVBS', 'TARGET-20-PARDYW', 'TARGET-20-PAREWJ', 'TARGET-20-PARYHL', 'TARGET-20-PASFKI', 'TARGET-20-PASGCJ', 'TARGET-20-PASGHD', 'TARGET-20-PASHUS', 'TARGET-20-PASLSE', 'TARGET-20-PASPIN', 'TARGET-20-PASZAI', 'TARGET-20-PATALD', 'TARGET-20-PATLDZ']

Split summary:
Training: 711 samples, 620 patients
Validation: 201 samples, 177 patients
Test: 100 samples, 89 patients


In [ ]:
# -------------------------
# 6. Add missingness indicators
# -------------------------
# Missingness itself can carry information in clinical datasets.

#This creates the list of features
# that will receive some form of imputation or missing-value handling.

# This can be useful because missingness patterns themselves may contain information about:

# which tests were ordered;
# historical diagnostic practices;
# patient severity;
# cohort differences.

IMPUTED_FEATURE_COLS = (
    BINARY_FEATURE_COLS
    + CATEGORICAL_FEATURE_COLS
    + ([CONDITIONAL_CONTINUOUS_COL]
       if CONDITIONAL_CONTINUOUS_COL in df.columns else [])
)

for split_df in [train_df, val_df, test_df]:
    for col in IMPUTED_FEATURE_COLS:
        split_df[f'{col}__WAS_MISSING'] = split_df[col].isna().astype('int8')

In [ ]:
# -------------------------
# 7. Fit imputers on training only
# -------------------------
# Binary genomic variables: use the most frequent observed 0/1 value learned
# from training. Missingness indicators preserve whether the value was absent.
binary_imputer = SimpleImputer(strategy='most_frequent')

if BINARY_FEATURE_COLS:
    train_df[BINARY_FEATURE_COLS] = binary_imputer.fit_transform(
        train_df[BINARY_FEATURE_COLS]
    )
    val_df[BINARY_FEATURE_COLS] = binary_imputer.transform(
        val_df[BINARY_FEATURE_COLS]
    )
    test_df[BINARY_FEATURE_COLS] = binary_imputer.transform(
        test_df[BINARY_FEATURE_COLS]
    )

    # Restore compact integer representation.
    for split_df in [train_df, val_df, test_df]:
        split_df[BINARY_FEATURE_COLS] = (
            split_df[BINARY_FEATURE_COLS].astype('int8')
        )


# High-missing categorical biomarker:
# preserve an explicit Missing category rather than pretending the most common
# biological result was observed.
if CATEGORICAL_FEATURE_COLS:
    categorical_imputer = SimpleImputer(
        strategy='constant',
        fill_value=CATEGORICAL_MISSING_LABEL
    )

    train_df[CATEGORICAL_FEATURE_COLS] = categorical_imputer.fit_transform(
        train_df[CATEGORICAL_FEATURE_COLS]
    )
    val_df[CATEGORICAL_FEATURE_COLS] = categorical_imputer.transform(
        val_df[CATEGORICAL_FEATURE_COLS]
    )
    test_df[CATEGORICAL_FEATURE_COLS] = categorical_imputer.transform(
        test_df[CATEGORICAL_FEATURE_COLS]
    )
else:
    categorical_imputer = None


In [ ]:
# -------------------------
# 8. Conditional FLT3 allelic-ratio handling
# -------------------------
# The allelic ratio is only clinically meaningful when FLT3_ITD_POSITIVE == 1.
# Therefore:
# - confirmed FLT3-negative samples receive ratio 0.0;
# - missing ratios among FLT3-positive samples are filled using the median
#   ratio learned from FLT3-positive TRAINING samples only.
if CONDITIONAL_CONTINUOUS_COL in df.columns:
    required_flt3_col = 'FLT3_ITD_POSITIVE'
    if required_flt3_col not in train_df.columns:
        raise KeyError(
            'FLT3_ITD_POSITIVE is required for conditional ratio handling.'
        )

    positive_train_ratios = train_df.loc[
        (train_df[required_flt3_col] == 1)
        & train_df[CONDITIONAL_CONTINUOUS_COL].notna(),
        CONDITIONAL_CONTINUOUS_COL
    ]

    if positive_train_ratios.empty:
        raise ValueError(
            'No observed FLT3 allelic ratios were found among FLT3-positive '
            'training samples.'
        )

    flt3_positive_ratio_median = float(positive_train_ratios.median())

    for split_df in [train_df, val_df, test_df]:
        negative_mask = split_df[required_flt3_col] == 0
        positive_missing_mask = (
            (split_df[required_flt3_col] == 1)
            & split_df[CONDITIONAL_CONTINUOUS_COL].isna()
        )

        split_df.loc[
            negative_mask,
            CONDITIONAL_CONTINUOUS_COL
        ] = 0.0

        split_df.loc[
            positive_missing_mask,
            CONDITIONAL_CONTINUOUS_COL
        ] = flt3_positive_ratio_median

    print(
        '\nTraining median FLT3-ITD allelic ratio among positive samples:',
        flt3_positive_ratio_median
    )
else:
    flt3_positive_ratio_median = None


Training median FLT3-ITD allelic ratio among positive samples: 0.55


In [ ]:
# -------------------------
# 9. Validate outputs
# -------------------------
MODELING_FEATURE_COLS = (
    BINARY_FEATURE_COLS
    + CATEGORICAL_FEATURE_COLS
    + ([CONDITIONAL_CONTINUOUS_COL]
       if CONDITIONAL_CONTINUOUS_COL in df.columns else [])
    + [f'{col}__WAS_MISSING' for col in IMPUTED_FEATURE_COLS]
)

for name, split_df in [
    ('Training', train_df),
    ('Validation', val_df),
    ('Test', test_df)
]:
    remaining = split_df[MODELING_FEATURE_COLS].isna().sum()
    remaining = remaining[remaining > 0]

    print(f'\n{name} remaining missing modeling values: {remaining.sum()}')
    if not remaining.empty:
        print(remaining.sort_values(ascending=False))

    assert remaining.sum() == 0

# Ensure the same modeling columns exist in every split.
assert set(MODELING_FEATURE_COLS).issubset(train_df.columns)
assert set(MODELING_FEATURE_COLS).issubset(val_df.columns)
assert set(MODELING_FEATURE_COLS).issubset(test_df.columns)



Training remaining missing modeling values: 0

Validation remaining missing modeling values: 0

Test remaining missing modeling values: 0


In [ ]:
# -------------------------
# 10. Save raw and imputed splits
# -------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Raw versions preserve original missingness and the same patient assignments.
# Recreate them from the original dataframe using final split patient sets.
train_raw_df = df[df['PATIENT_ID'].isin(train_patients)].copy()
val_raw_df = df[df['PATIENT_ID'].isin(val_patients)].copy()
test_raw_df = df[df['PATIENT_ID'].isin(test_patients)].copy()

train_raw_df.to_csv(OUTPUT_DIR / 'sample_train_raw.csv', index=False)
val_raw_df.to_csv(OUTPUT_DIR / 'sample_validation_raw.csv', index=False)
test_raw_df.to_csv(OUTPUT_DIR / 'sample_test_raw.csv', index=False)

train_df.to_csv(OUTPUT_DIR / 'sample_train_imputed.csv', index=False)
val_df.to_csv(OUTPUT_DIR / 'sample_validation_imputed.csv', index=False)
test_df.to_csv(OUTPUT_DIR / 'sample_test_imputed.csv', index=False)

# Save fitted preprocessing objects.
joblib.dump(
    binary_imputer,
    OUTPUT_DIR / 'sample_binary_imputer.joblib'
)

if categorical_imputer is not None:
    joblib.dump(
        categorical_imputer,
        OUTPUT_DIR / 'sample_categorical_imputer.joblib'
    )

# Save split membership explicitly for reproducibility.
pd.DataFrame({'PATIENT_ID': sorted(train_patients)}).to_csv(
    OUTPUT_DIR / 'sample_train_patient_ids.csv', index=False
)
pd.DataFrame({'PATIENT_ID': sorted(val_patients)}).to_csv(
    OUTPUT_DIR / 'sample_validation_patient_ids.csv', index=False
)
pd.DataFrame({'PATIENT_ID': sorted(test_patients)}).to_csv(
    OUTPUT_DIR / 'sample_test_patient_ids.csv', index=False
)

metadata = {
    'random_state': RANDOM_STATE,
    'aligned_with_patient_splits': ALIGN_WITH_PATIENT_SPLITS,
    'id_columns': ID_COLS,
    'metadata_columns_not_used_as_modeling_features': METADATA_COLS,
    'binary_feature_columns': BINARY_FEATURE_COLS,
    'categorical_feature_columns': CATEGORICAL_FEATURE_COLS,
    'conditional_continuous_column': CONDITIONAL_CONTINUOUS_COL,
    'flt3_positive_training_ratio_median': flt3_positive_ratio_median,
    'missingness_indicator_columns': [
        f'{col}__WAS_MISSING' for col in IMPUTED_FEATURE_COLS
    ],
    'excessive_missingness_threshold': EXCESSIVE_MISSINGNESS_THRESHOLD,
    'excluded_excessively_missing_columns': EXCESSIVELY_MISSING_COLS,
    'modeling_feature_columns': MODELING_FEATURE_COLS,
    'n_train_samples': len(train_df),
    'n_validation_samples': len(val_df),
    'n_test_samples': len(test_df),
    'n_train_patients': train_df['PATIENT_ID'].nunique(),
    'n_validation_patients': val_df['PATIENT_ID'].nunique(),
    'n_test_patients': test_df['PATIENT_ID'].nunique(),
    'unassigned_sample_patients': unassigned_sample_patients
}

with open(
    OUTPUT_DIR / 'sample_preprocessing_metadata.json',
    'w',
    encoding='utf-8'
) as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

print('\nSaved outputs to:', OUTPUT_DIR)
print('\nGenerated files:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print('-', path.name)




Saved outputs to: /content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Sample/sample_split_and_imputed

Generated files:
- sample_binary_imputer.joblib
- sample_preprocessing_metadata.json
- sample_test_imputed.csv
- sample_test_patient_ids.csv
- sample_test_raw.csv
- sample_train_imputed.csv
- sample_train_patient_ids.csv
- sample_train_raw.csv
- sample_validation_imputed.csv
- sample_validation_patient_ids.csv
- sample_validation_raw.csv


In [ ]:
import pandas as pd

# CSV'yi oku
df = pd.read_csv("/content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Sample/"
    "sample_split_and_imputed/sample_validation_imputed.csv")
# veya kendi dosya yolunu kullan

# SOMATIC_STATUS sütununun indeksini bul
somatic_idx = df.columns.get_loc("SOMATIC_STATUS")

# SOMATIC_STATUS'tan sonraki tüm sütunları kaldır
df = df.iloc[:, :somatic_idx + 1]

# Kaydet
df.to_csv(
    "/content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Sample/"
    "sample_split_and_imputed/clean/sample_validation_imputed_clean.csv",
    index=False
)

print(df.columns)

Index(['PATIENT_ID', 'SAMPLE_ID', 'T_6_9', 'T_8_21', 'T_3_5_Q25_Q34',
       'T_6_11_Q27_Q23', 'T_9_11_P22_Q23', 'T_10_11_P11_2_Q23',
       'T_11_19_Q23_P13_1', 'INV_16', 'DEL_5Q', 'DEL_7Q', 'DEL_9Q',
       'MONOSOMY_5', 'MONOSOMY_7', 'TRISOMY_8', 'TRISOMY_21', 'MLL', 'MINUS_Y',
       'MINUS_X', 'FLT3_ITD_POSITIVE', 'FLT3_ITD_ALLELIC_RATIO', 'FLT3_PM',
       'NPM_MUTATION', 'CEBPA_MUTATION', 'WT1_MUTATION', 'ONCOTREE_CODE',
       'ANALYSIS_COHORT', 'CANCER_TYPE', 'CANCER_TYPE_DETAILED',
       'SOMATIC_STATUS'],
      dtype='object')
